# Stickverse Master Studio — Colab Starter\nFour channel presets. This is an early **illustrated slideshow prototype**, not a finished animated production studio. Narration uses offline synthetic espeak-ng. Paste your own original English script; script length controls video duration.\nRun this notebook with **Runtime → Run all**. CPU runtime is sufficient.\n

In [ ]:
import os, subprocess, textwrap, math, json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
from google.colab import files

CHANNELS = {
 "1": ("Dark Psychology", (12,22,35)),
 "2": ("Mystery & Horror", (30,12,32)),
 "3": ("Stickman What-If", (18,40,55)),
 "4": ("Strange History", (52,38,23)),
}
print("CHANNELS:", CHANNELS)
choice = input("Channel number (1-4) [3]: ").strip() or "3"
if choice not in CHANNELS: choice="3"
title = input("Video title: ").strip() or "What If Gravity Disappeared?"
print("Paste your English narration script in the next input. For 6-15 minutes use about 850-2100 words.")
script = input("Script: ").strip()
if not script: script = (("Imagine the world changing in a single second. What would happen next? "
 "We explore the consequences, step by step, and discover why the smallest changes can reshape everything. ") * 10)
out=Path("/content/stickverse_output"); out.mkdir(exist_ok=True)
(out/"script.txt").write_text(script)
name,bg=CHANNELS[choice]
# Offline narration fallback; replaces paid voice APIs, but sounds synthetic.
subprocess.run(["apt-get","-qq","update"],check=True,stdout=subprocess.DEVNULL)
subprocess.run(["apt-get","-qq","install","-y","espeak-ng","ffmpeg"],check=True,stdout=subprocess.DEVNULL)
wav=str(out/"voice.wav")
subprocess.run(["espeak-ng","-v","en-us","-s","145","-w",wav,script],check=True)
probe=subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","default=nw=1:nk=1",wav],text=True)
duration=float(probe.strip())
# Generate one illustrated frame per ~8 seconds and add gentle zoom/pan in ffmpeg.
font="/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
f=ImageFont.truetype(font,52); small=ImageFont.truetype(font,31)
sentences=[s.strip() for s in __import__("re").split(r"(?<=[.!?])\\s+",script) if s.strip()]
if not sentences: sentences=[script]
scene_count=max(1,math.ceil(duration/8))
for i in range(scene_count):
    im=Image.new("RGB",(1280,720),bg); d=ImageDraw.Draw(im)
    d.rectangle((35,35,1245,685),outline=(85,165,180),width=4)
    d.text((65,62),name,font=small,fill=(150,220,230))
    d.text((65,135),textwrap.fill(title,31),font=f,fill="white",spacing=12)
    # Animated stickman pose changes each scene
    x=750+(i%5-2)*35; y=390; glow=(70,220,225)
    d.ellipse((x-36,y-120,x+36,y-48),outline=glow,width=9)
    d.line((x,y-45,x,y+105),fill=glow,width=12)
    d.line((x,y,x-90,y+(-40 if i%2 else 60)),fill=glow,width=11)
    d.line((x,y,x+90,y+(60 if i%2 else -40)),fill=glow,width=11)
    d.line((x,y+105,x-70,y+210),fill=glow,width=12)
    d.line((x,y+105,x+70,y+210),fill=glow,width=12)
    excerpt=sentences[min(len(sentences)-1,int(i*len(sentences)/scene_count))]
    d.text((65,555),textwrap.fill(excerpt[:100],52),font=small,fill=(220,220,220),spacing=6)
    im.save(out/f"frame_{i:04d}.png")
# Render at 720p for reasonable Colab runtime. 16:9 H264 MP4.
concat=out/"frames.txt"
with concat.open("w") as h:
    for i in range(scene_count):
        h.write(f"file '{out/f'frame_{i:04d}.png'}'\\n")
        h.write("duration 8\\n")
    h.write(f"file '{out/f'frame_{scene_count-1:04d}.png'}'\\n")
video=str(out/"final_video.mp4")
subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-y","-safe","0","-f","concat","-i",str(concat),"-i",wav,"-t",str(duration),"-c:v","libx264","-preset","ultrafast","-pix_fmt","yuv420p","-r","24","-c:a","aac","-b:a","128k","-movflags","+faststart",video],check=True)
print(f"READY: {video} | {duration/60:.1f} minutes | {scene_count} scenes")
display(Video(video,embed=False))
files.download(video)
